In [2]:
"""
SVM comparison run — trains on svm_train_80.xlsx, predicts on
eval_holdout_20_unlabeled.csv, saves preds_svm.csv in the same format as
preds_zeroshot.csv / preds_fewshot.csv (id, predicted_label) so all three
can be scored against eval_holdout_20_ground_truth.csv the same way.

Mirrors the modeling choices from phase1_SVMclassification.ipynb (TF-IDF
unigrams-to-trigrams, linear vs rbf SVM, pick best by weighted F1) but:
  - points DATA_PATH at svm_train_80.xlsx instead of the full labeled file
  - points the prediction step at eval_holdout_20_unlabeled.csv instead of
    master_posts.csv (no chunking needed — only 82 rows)
  - saves model/vectorizer/confusion matrix into a separate comparison_run
    folder so this doesn't overwrite your production model files
  - writes predictions as preds_svm.csv (id, predicted_label,
    prediction_confidence) instead of the *_classified.csv naming

IMPORTANT: the confusion matrix / accuracy this script prints is computed
on an INTERNAL 80/20 split of svm_train_80.xlsx (i.e. it further splits
your training pool just to sanity-check the model). That is NOT the number
to report for the SVM-vs-zeroshot-vs-fewshot comparison — that comes later,
from scoring preds_svm.csv against eval_holdout_20_ground_truth.csv.
"""

import re
import pickle
import warnings
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")  # no display needed on HPC
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import SVC
from sklearn.metrics import (
    classification_report, confusion_matrix,
    accuracy_score, f1_score
)

warnings.filterwarnings("ignore")

# ── CONFIG — edit these paths ────────────────────────────────────────────────
DATA_PATH = "svm_train_80.csv"
EVAL_UNLABELED_PATH = "eval_holdout_20_unlabeled.csv"

OUT_DIR = Path("/Users/nadia/Desktop/redditRun_june/classification_redo/svm")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_SAVE_PATH = OUT_DIR / "svm_trigram_model.pkl"
VECTORIZER_SAVE_PATH = OUT_DIR / "svm_trigram_vectorizer.pkl"
CONFUSION_MATRIX_PATH = OUT_DIR / "svm_trigram_confusion_matrix_INTERNAL_VALIDATION.png"
PREDS_OUTPUT_PATH = OUT_DIR / "preds_svm.csv"

NGRAM_TYPE = "unigrams_to_trigrams"  # 'trigrams_only' or 'unigrams_to_trigrams'
TEST_SIZE = 0.2       # internal validation split of svm_train_80.xlsx, not the eval set
RANDOM_STATE = 42


# ── Preprocessing (same as original notebook) ────────────────────────────────
def preprocess_text(text):
    text = str(text).lower()
    text = re.sub(r"http\S+|www.\S+", "", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


def handle_multilabels(df):
    multi_label_mask = df["label"].astype(str).str.contains(",")
    n = multi_label_mask.sum()
    if n > 0:
        print(f"  Found {n} multi-label samples — taking first label")
        df["label"] = df["label"].astype(str).apply(lambda x: x.split(",")[0])
    df["label"] = df["label"].astype(int)
    return df


def ngram_range_from_config():
    return (3, 3) if NGRAM_TYPE == "trigrams_only" else (1, 3)


# ── 1. Load + preprocess training pool ───────────────────────────────────────
print("[1/6] Loading training data...")
df = pd.read_csv(DATA_PATH)
print(f"  Loaded {len(df)} samples from {DATA_PATH}")

df["text_processed"] = df["text"].apply(preprocess_text)
if "Label" in df.columns:
    df.rename(columns={"Label": "label"}, inplace=True)
df = handle_multilabels(df)
print("  Label distribution:")
print(df["label"].value_counts().sort_index())

# ── 2. Internal train/test split — sanity check only, NOT the eval set ──────
print(f"\n[2/6] Internal {int((1-TEST_SIZE)*100)}/{int(TEST_SIZE*100)} split of svm_train_80.xlsx "
      f"(for validation metrics only)...")
X = df["text_processed"].values
y = df["label"].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)
print(f"  Internal train: {len(X_train)} | Internal test: {len(X_test)}")

# ── 3. Feature extraction ────────────────────────────────────────────────────
print(f"\n[3/6] Extracting TF-IDF features ({NGRAM_TYPE})...")
vectorizer = TfidfVectorizer(
    max_features=1000,
    min_df=2,
    max_df=0.8,
    ngram_range=ngram_range_from_config(),
    stop_words="english",
)
X_train_tfidf = vectorizer.fit_transform(X_train)
X_test_tfidf = vectorizer.transform(X_test)
print(f"  Train matrix: {X_train_tfidf.shape} | Test matrix: {X_test_tfidf.shape}")

# ── 4. Train SVM (linear + rbf), pick best by weighted F1 ──────────────────
print("\n[4/6] Training SVM (linear + rbf kernels)...")
results = {}
for kernel in ["linear", "rbf"]:
    model = SVC(
        kernel=kernel,
        C=1.0,
        random_state=RANDOM_STATE,
        class_weight="balanced",
        probability=True,
    )
    model.fit(X_train_tfidf, y_train)
    y_pred = model.predict(X_test_tfidf)

    results[kernel] = {
        "model": model,
        "accuracy": accuracy_score(y_test, y_pred),
        "f1_weighted": f1_score(y_test, y_pred, average="weighted"),
        "f1_macro": f1_score(y_test, y_pred, average="macro"),
        "predictions": y_pred,
    }
    print(f"  {kernel:>6}: accuracy={results[kernel]['accuracy']:.4f}  "
          f"f1_weighted={results[kernel]['f1_weighted']:.4f}")

best_kernel = max(results, key=lambda k: results[k]["f1_weighted"])
best_model = results[best_kernel]["model"]
best_predictions = results[best_kernel]["predictions"]
print(f"\n  Best kernel: {best_kernel} (f1_weighted={results[best_kernel]['f1_weighted']:.4f})")

# ── 5. Internal validation report + confusion matrix (NOT the comparison metric) ──
print(f"\n[5/6] Internal validation report (on the held-back 20% of svm_train_80.xlsx):")
print(classification_report(y_test, best_predictions))

cm = confusion_matrix(y_test, best_predictions)
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues")
plt.title(f"Internal Validation — SVM {best_kernel} kernel (NOT the eval-holdout comparison)")
plt.ylabel("True Label")
plt.xlabel("Predicted Label")
plt.savefig(CONFUSION_MATRIX_PATH, dpi=300, bbox_inches="tight")
plt.close()
print(f"  Saved: {CONFUSION_MATRIX_PATH}")

with open(MODEL_SAVE_PATH, "wb") as f:
    pickle.dump(best_model, f)
with open(VECTORIZER_SAVE_PATH, "wb") as f:
    pickle.dump(vectorizer, f)
print(f"  Saved model: {MODEL_SAVE_PATH}")
print(f"  Saved vectorizer: {VECTORIZER_SAVE_PATH}")

# ── 6. Predict on the real eval-holdout set ──────────────────────────────────
print(f"\n[6/6] Predicting on eval-holdout set...")
eval_df = pd.read_csv(EVAL_UNLABELED_PATH)
if "text" not in eval_df.columns:
    raise ValueError("Eval file must contain a 'text' column.")
print(f"  Loaded {len(eval_df)} rows from {EVAL_UNLABELED_PATH}")

eval_df["text_processed"] = eval_df["text"].apply(preprocess_text)
X_eval_tfidf = vectorizer.transform(eval_df["text_processed"].values)

eval_df["predicted_label"] = best_model.predict(X_eval_tfidf)
probabilities = best_model.predict_proba(X_eval_tfidf)
eval_df["prediction_confidence"] = np.max(probabilities, axis=1)

preds_out = eval_df[["id", "text", "predicted_label", "prediction_confidence"]].copy()
preds_out.to_csv(PREDS_OUTPUT_PATH, index=False)

print(f"\nSaved: {PREDS_OUTPUT_PATH}")
print(f"Predicted label distribution:\n{preds_out['predicted_label'].value_counts()}")
print(f"\nDone. This file is ready to be scored against eval_holdout_20_ground_truth.csv, "
      f"the same way as preds_zeroshot.csv and preds_fewshot.csv.")

[1/6] Loading training data...
  Loaded 324 samples from svm_train_80.csv
  Label distribution:
label
0    166
1    158
Name: count, dtype: int64

[2/6] Internal 80/20 split of svm_train_80.xlsx (for validation metrics only)...
  Internal train: 259 | Internal test: 65

[3/6] Extracting TF-IDF features (unigrams_to_trigrams)...
  Train matrix: (259, 1000) | Test matrix: (65, 1000)

[4/6] Training SVM (linear + rbf kernels)...
  linear: accuracy=0.7692  f1_weighted=0.7686
     rbf: accuracy=0.7692  f1_weighted=0.7686

  Best kernel: linear (f1_weighted=0.7686)

[5/6] Internal validation report (on the held-back 20% of svm_train_80.xlsx):
              precision    recall  f1-score   support

           0       0.75      0.82      0.78        33
           1       0.79      0.72      0.75        32

    accuracy                           0.77        65
   macro avg       0.77      0.77      0.77        65
weighted avg       0.77      0.77      0.77        65

  Saved: /Users/nadia/Deskto